In [1]:
import yfinance as yf
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense, GRU, Input

In [3]:
end = datetime.now()
start = datetime(end.year - 10, end.month, end.day)
stock = "^NSEBANK"
bit_coin_data = yf.download(stock, start, end)

[*********************100%***********************]  1 of 1 completed


In [5]:
# Extract Close Price
Closing_price = bit_coin_data[['Close']]

In [7]:
# Normalize Data
scaler = MinMaxScaler(feature_range=(0,1))
scaled_data = scaler.fit_transform(Closing_price[['Close']].values)

In [9]:
# Prepare Data
x_data, y_data = [], []
base_days = 100
for i in range(base_days, len(scaled_data)):
    x_data.append(scaled_data[i-base_days:i])
    y_data.append(scaled_data[i])

x_data, y_data = np.array(x_data), np.array(y_data)

In [11]:
# Train-Test Split
len_train = int(len(x_data) * 0.9)
x_train, y_train = x_data[:len_train], y_data[:len_train]
x_test, y_test = x_data[len_train:], y_data[len_train:]

In [13]:
# Build GRU Model
model_gru = Sequential([
    Input(shape=(x_train.shape[1], 1)),
    GRU(128, return_sequences=True),
    GRU(64, return_sequences=False),
    Dense(25),
    Dense(1)
])


In [15]:
# Compile & Train Model
model_gru.compile(optimizer='adam', loss='mean_squared_error')
model_gru.fit(x_train, y_train, batch_size=5, epochs=10)

Epoch 1/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - loss: 0.0053
Epoch 2/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 14s 38ms/step - loss: 2.7703e-04
Epoch 3/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 14s 38ms/step - loss: 2.3610e-04
Epoch 4/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 14s 38ms/step - loss: 3.0287e-04
Epoch 5/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 14s 38ms/step - loss: 2.1788e-04
Epoch 6/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 14s 38ms/step - loss: 1.9230e-04
Epoch 7/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 14s 38ms/step - loss: 2.4011e-04
Epoch 8/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 14s 38ms/step - loss: 1.4845e-04
Epoch 9/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 14s 38ms/step - loss: 1.5286e-04
Epoch 10/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 15s 39ms/step - loss: 2.1114e-04


In [17]:
# Predict and Inverse Transform
predictions_gru = model_gru.predict(x_test)
inv_predictions_gru = scaler.inverse_transform(predictions_gru)
inv_y_test_gru = scaler.inverse_transform(y_test)

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 92ms/step


In [19]:
model_gru.save("gru_model.keras")